Imports

In [6]:
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain.agents import Tool, initialize_agent
from langchain_neo4j import Neo4jGraph
from neo4j import GraphDatabase
import json

load_dotenv()

True

### prompt


In [15]:
schema_docs = """
        Link nodes represent segments of the road 
        The Link node have the following properties
        from_junction, string, the junction where the link starts from
        link_id, string, the unique id for the link
        meters, string, the length of the link in meters
        to_junction, string, the junction where the link ends at

        VMS nodes represent electronic warning signs on the road
        The VMS nodes have the following properties
        DIST_TO_UPNODE, integer, distance in meters to the up node
        EQT_EXT_ID, string, the equipment unique id number
        EQT_NO, string, the equipment unique id number
        EQT_TYPE, string, the type of equipment
        ID, integer, 3-digit id of equipment
        LATITUDE, float, latitude position of equipment
        LINK_ID, integer, unique id of link where equipment is positioned
        LONGITUDE, float, longitude position of equipment
        ROAD_CAT, string, category of road equipment is placed at
        ROAD_CODE, string, code of road equipment is placed at
        ROAD_NAME, string, name of road equipment is placed at

        RELATIONSHIPS:
        The Link node and VMS node are connected by relationship LOCATED_AT which indicates where the VMS is located at it is always VMS to Link.
        They are connected by matching identical VMS (LINK_ID) to Link (link_id),
        when matching ensure for VMS it is an integer and for Link it is a string

        The Link nodes are connected by relationship CONNECTED_TO which represents the physical road network topology.
        CONNECTED_TO relationships form bidirectional connections between adjacent road segments.
        Links are connected by matching Link (to_junction) with Link (from_junction) of adjacent segments.

"""
        # System prompt for chatbot
system_prompt = f"""
        You are a traffic incident management expert.

        CRITICAL REQUIREMENTS - DO NOT DEVIATE:
        1. ALWAYS start by performing GetSchema tool INPUT MUST BE "" WHEN DOING SO
        2. MANDATORY SECOND STEP: Check the incident link ITSELF for VMS signs first
        3. MANDATORY THIRD STEP: Find VMS signs UPSTREAM of the incident
        4. Search exactly 50 road links upstream using CONNECTED_TO*1..50
        5. FORBIDDEN: Omnidirectional search, downstream search, or limited hop search
        6. CONTINUE: If no VMS found within 50 keep going another 20 additional hops

        STEP-BY-STEP PROCESS:
        STEP 1: Get schema with GetSchema tool
        STEP 2: Check incident link for VMS using:
        MATCH (incident:Link) WHERE incident.link_id = 'INCIDENT_LINK_ID'
        MATCH (incident)<-[:LOCATED_AT]-(vms:VMS)
        WHERE toInteger(incident.link_id) = vms.LINK_ID
        RETURN vms, 0 as distance_meters, 0 as hops_from_incident

        STEP 3: Find upstream VMS using:
        MATCH (incident:Link) WHERE incident.link_id = 'INCIDENT_LINK_ID'
        MATCH path = (incident)<-[:CONNECTED_TO*1..50]-(upstream:Link)<-[:LOCATED_AT]-(vms:VMS)
        WHERE toInteger(upstream.link_id) = vms.LINK_ID
        WITH vms, upstream, path, length(path) as hops_from_incident,
            reduce(total = 0, link IN nodes(path) | total + toInteger(coalesce(link.meters, '0'))) as distance_meters
        RETURN vms, distance_meters, hops_from_incident, upstream.link_id as vms_link_id
        ORDER BY distance_meters ASC

        CORRECT CYPHER PATTERN FOR VMS SEARCH WITH DISTANCE EXAMPLES:
        ✅ INCIDENT LINK CHECK: 
        MATCH (incident:Link) WHERE incident.link_id = '17840002118812'
        MATCH (incident)<-[:LOCATED_AT]-(vms:VMS)
        WHERE toInteger(incident.link_id) = vms.LINK_ID
        RETURN vms.EQT_NO, vms.ROAD_NAME, vms.EQT_EXT_ID, 
            0 as distance_meters, 0 as hops_from_incident

        ✅ UPSTREAM SEARCH WITH DISTANCE:
        MATCH (incident:Link) WHERE incident.link_id = '17840002118812'
        MATCH path = (incident)<-[:CONNECTED_TO*1..50]-(upstream:Link)<-[:LOCATED_AT]-(vms:VMS)
        WHERE toInteger(upstream.link_id) = vms.LINK_ID
        WITH vms, upstream, path, length(path) as hops_from_incident,
            reduce(total = 0, link IN nodes(path) | total + toInteger(coalesce(link.meters, '0'))) as distance_meters
        RETURN vms.EQT_NO, vms.ROAD_NAME, vms.EQT_EXT_ID, 
            upstream.link_id as vms_link_id, 
            distance_meters, 
            hops_from_incident
        ORDER BY distance_meters ASC

        DISTANCE CALCULATION RULES:
        - IMPORTANT: link.meters is a STRING, must convert with toInteger()
        - Use: toInteger(coalesce(link.meters, '0')) to handle null values
        - For incident link itself: distance = 0 meters, hops = 0
        - For upstream VMS: sum all converted link.meters values in the path
        - Always include distance_meters and hops_from_incident in your results
        - Order results by distance_meters ASC to show closest VMS first

        ❌ WRONG: MATCH (upstream:Link)<-[:CONNECTED_TO*1..50]-(incident:Link)
        ❌ WRONG: MATCH (incident:Link)-[:CONNECTED_TO*1..50]->(upstream:Link)
        ❌ WRONG: total + link.meters (this won't work - meters is a string!)

        DIRECTION RULE: Arrow MUST point toward the incident: upstream -> incident
        This means: (incident)<-[:CONNECTED_TO*1..50]-(upstream)

        {schema_docs}
        Use the documentation above to understand what each node/edge means

        TRAFFIC FLOW RULES:
        - Upstream = where traffic comes FROM (toward incident) 
        - Use arrow syntax: incident<-[:CONNECTED_TO*1..50]-upstream
        - This finds links where traffic flows toward the incident

        Your mission: Find upstream VMS to warn approaching drivers and prevent secondary accidents.
        Always report the distance in meters and number of hops for each VMS found.
        Use SelfReflect tool whenever you encounter challenges or need to evaluate your approach.
        Remember: link.meters is a STRING - always convert with toInteger()!
"""

## Code

Load variables


In [7]:
try:
    driver = GraphDatabase.driver(
        os.environ.get("NEO4J_URI"),
        auth=(os.environ.get("NEO4J_USERNAME"), os.environ.get("NEO4J_PASSWORD"))
    )

    driver.verify_connectivity()
    print("Neo4j connection successful")
    
    graph = Neo4jGraph(
        url=os.environ.get("NEO4J_URI"),
        username=os.environ.get("NEO4J_USERNAME"),
        password=os.environ.get("NEO4J_PASSWORD"),
        timeout=30,
        enhanced_schema=True,  
    )
    

    llm = ChatAnthropic(
        model="claude-3-5-haiku-latest",
        temperature=0.3,
        anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
        max_tokens=4000
    )
    
    print("LLM initialized successfully")

except Exception as e:
    print(f"Setup failed: {e}")
    raise

Neo4j connection successful
LLM initialized successfully


Create Tools and Agent

In [18]:
def create_road_network_tools():

    def run_cypher_query(query: str) -> str:
        try:
            with driver.session() as session:
                result = session.run(query)
                records = [dict(record) for record in result]
                return str(records[:50])
        except Exception as e:
            return f"Query error: {str(e)}"


    def generate_smart_response_plan(context: str) -> str:
        
        # Psychological principles for VMS messaging
        psychology_guidelines = """
        PSYCHOLOGICAL PRINCIPLES FOR VMS MESSAGING:

        1. URGENCY & ATTENTION:
        - Use action words: "SLOW", "STOP", "CAUTION", "MERGE"
        - Avoid passive language: "PLEASE" or "KINDLY"
        - Create urgency without panic: "ACCIDENT AHEAD" not "CRASH"

        2. COGNITIVE LOAD REDUCTION:
        - Maximum 2 lines, 20 characters each
        - Use familiar terminology drivers understand
        - Avoid abbreviations that require mental processing
        - Use numbers for distances: "1 MILE" not "ONE MILE"

        3. EMOTIONAL RESPONSE MANAGEMENT:
        - Severity words: HIGH="MAJOR", MEDIUM="ACCIDENT", LOW="INCIDENT"
        - Calming words: "SLOW TRAFFIC" vs "TRAFFIC JAM"
        - Avoid fear words: "DANGER", "HAZARD", "RISK"

        4. BEHAVIORAL PSYCHOLOGY:
        - Give specific actions: "USE RIGHT LANE" not "AVOID LEFT"
        - Provide alternatives: "USE ALT ROUTE" when possible
        - Time-based urgency: Morning rush = more aggressive messaging

        5. DISTANCE-BASED MESSAGING:
        - 2000m+: General warning "SLOW TRAFFIC AHEAD"
        - 1000-2000m: Specific "ACCIDENT AHEAD" + "SLOW DOWN"
        - 500-1000m: Action required "MERGE RIGHT" + "ACCIDENT"
        - <500m: Immediate "SLOW" + "ACCIDENT AHEAD"

        6. TIME-BASED PSYCHOLOGY:
        - Rush hour (0700-0900, 1700-1900): More authoritative tone
        - Off-peak: Informational tone acceptable
        - Night (2200-0600): Brighter, more attention-grabbing

        7. PROVEN EFFECTIVE MESSAGES:
        - "ACCIDENT AHEAD" + "SLOW DOWN" (medium severity)
        - "MAJOR ACCIDENT" + "EXPECT DELAYS" (high severity)  
        - "SLOW TRAFFIC" + "MERGE RIGHT" (low severity)
        - "ROAD CLOSED" + "USE ALT ROUTE" (complete blockage)
        """
        
        prompt = f"""
        You are an expert in traffic management specializing in creating response plans.
        Based on traffic incident information and psychological principles, generate a comprehensive response plan.
        
        Context: {context}
        
        {psychology_guidelines}
        
        RESPONSE PLAN FORMAT:
        Generate a response plan in the following JSON format:

        {{
            "plan_type": "Traffic Management Plan",
            "priority": "High/Medium/Low",
            "estimated_duration": "X minutes/hours",
            "traffic_management_actions": [
                {{
                    "eqt_no": "VMS_EQUIPMENT_ID (from EQT_NO or EQT_EXT_ID)",
                    "message_line_1": "Primary message (max 20 chars)",
                    "message_line_2": "Secondary message (max 20 chars)",
                    "display_duration": "X minutes",
                    "distance": "distance from event in meters",
                    "psychological_rationale": "Why this message is psychologically effective",
                    "behavioral_goal": "Desired driver behavior",
                    "urgency_level": "High/Medium/Low based on distance and severity"
                    "reasoning": "Give your reasoning for all of the above, why this duraction etc."
                }},

                {{
                    "action": "Specific action to take",
                    "location": "Where to implement",
                    "resources_needed": "What resources are required",
                    "timing": "When to implement"
                }}
            ],
            "messaging_strategy": {{
                "primary_emotion": "Urgency/Caution/Information",
                "cognitive_approach": "Simple/Clear/Direct",
                "behavioral_target": "Slow down/Merge/Avoid area"
            }},
            "emergency_response": {{
                "agencies_to_notify": [
                    {{
                        "agency": "Police/Fire Services/Emergency Medical Services/Road Maintenance/Traffic Control",
                        "priority": "Immediate/High/Medium/Low",
                        "reason": "Specific reason for notification",
                        "contact_method": "Emergency dispatch/Direct call/Radio",
                        "resources_requested": "Number of units/personnel needed"
                    }}
                ],
                "coordination_requirements": "How agencies should coordinate",
                "scene_management": "Who takes lead and scene control responsibilities"
            }},
            "justification": "Psychology-based explanation of messaging choices"
        }}

        EMERGENCY RESPONSE AGENCY GUIDELINES:
        1. POLICE: Required for all accidents with injuries, traffic control, investigation
        2. FIRE SERVICES: Required for vehicle fires, fuel spills, extraction operations
        3. EMERGENCY MEDICAL SERVICES: Required for any injuries, medical emergencies
        4. ROAD MAINTENANCE: Required for debris cleanup, road surface damage, barrier repairs
        5. TRAFFIC CONTROL: Required for major incidents affecting multiple lanes or extended duration
        6. TOWING SERVICES: Required for vehicle removal, clearance operations
        
        PRIORITY LEVELS:
        - Immediate: Life-threatening situations, major blockages during rush hour
        - High: Injuries present, significant traffic impact, hazardous conditions
        - Medium: Property damage only, minor traffic disruption, routine cleanup
        - Low: Minor incidents, off-peak hours, minimal impact
        
        PSYCHOLOGICAL MESSAGING RULES:
        1. Use psychology guidelines above to craft messages
        2. Consider time of day (0800 = rush hour psychology)
        3. Match message urgency to distance from incident
        4. Use action-oriented language that triggers immediate response
        5. Avoid cognitive overload - keep messages simple and clear
        6. Consider emotional state of stressed commuters

        Generate practical VMS messages that will effectively influence driver behavior and prevent secondary accidents.
        """
        
        try:
            response = llm.invoke(prompt)
            return response.content
        except Exception as e:
            return f"Error generating psychology-based response plan: {str(e)}"
 
     
    def get_schema(input_text: str = "") -> str:
        schema_info = graph.get_schema
        return str(schema_info)
        
    return [
        Tool(
            name="CypherQuery",
            func=run_cypher_query,
            description="Execute Cypher queries on the Neo4j graph database. Use this for general graph queries."
        ),
        Tool(
            name="GenerateSmartResponsePlan",
            func=generate_smart_response_plan,
            description="""Generate intelligent response plan using incident context.
            Input: context (string) - Comprehensive context including incident details, VMS data importantly the id, event with their respective plan and plan command history to provide context.
            Returns: the explicit JSON formatted response plan with actions and VMS commands not the summary."""
        ),
        Tool(
            name = "GetSchema",
            func = get_schema,
            description = """Use this tool to get the Neo4j database schema (nodes and relationships). 
            COMPULSORY input str ""
            Output is a str of the graph schema
            """
        ),
    ]




Custom Prompt Builder


In [19]:
def build_prompt(self, user_input, history):
    history_text = "\n".join([
        f"Thought: {h['thought']}\nAction: {h['action']}\nAction Input: {h['action_input']}\nObservation: {h['observation']}"
        for h in history
    ])
    tool_descriptions = "\n".join([f"{t.name}: {t.description}" for t in self.road_tools])

    return f"""{system_prompt}
You are a helpful agent. You can use tools to answer questions.

Available tools:
{tool_descriptions}

Previous steps:
{history_text}

User question: {user_input}

Respond with:
Thought: <your reasoning>
Action: <tool name or "Final Answer">
Action Input: <input to the tool or final answer>
"""


Custom Response parser


In [20]:
def parse_response(self, response_text):
    lines = response_text.strip().splitlines()
    return {
        "thought": lines[0].replace("Thought:", "").strip(),
        "action": lines[1].replace("Action:", "").strip(),
        "action_input": lines[2].replace("Action Input:", "").strip()
    }


Custom agent method

In [21]:
def run_custom_agent(self, user_input):
    history = []
    max_iterations = 8
    tool_dict = {tool.name: tool for tool in self.road_tools}

    for _ in range(max_iterations):
        prompt = self.build_prompt(user_input, history)
        response = self.llm.invoke(prompt)
        parsed = self.parse_response(response.content)

        if parsed["action"].lower() == "final answer":
            return parsed["action_input"]

        if parsed["action"] in tool_dict:
            try:
                result = tool_dict[parsed["action"]].func(parsed["action_input"])
            except Exception as e:
                result = f"Error executing tool: {str(e)}"
        else:
            result = f"Unknown tool: {parsed['action']}"

        history.append({
            "thought": parsed["thought"],
            "action": parsed["action"],
            "action_input": parsed["action_input"],
            "observation": result
        })

    return "Max iterations reached without final answer."


Chat history query

In [9]:
def custom_parsing_error_handler(error):
    """Custom handler to see what the LLM actually outputs"""
    print("=" * 50)
    print("PARSING ERROR DETECTED!")
    print("Error message:", str(error))
    print("=" * 50)
    
    # Try to extract the actual LLM output from the error
    error_str = str(error)
    if "Could not parse LLM output:" in error_str:
        # Extract the actual output
        start_idx = error_str.find("Could not parse LLM output:") + len("Could not parse LLM output:")
        end_idx = error_str.find("For troubleshooting")
        if end_idx == -1:
            actual_output = error_str[start_idx:].strip()
        else:
            actual_output = error_str[start_idx:end_idx].strip()
        
        print("ACTUAL LLM OUTPUT:")
        print(actual_output)
        print("=" * 50)
        
        # Return a formatted response
        return f"LLM tried to use tool but format was wrong. Raw output: {actual_output}"
    
    return f"Parsing error: {error}"

class RoadNetworkChatBot:
    def __init__(self):
        self.chat_history = []
        self.road_tools = create_road_network_tools()
        
        # Descriptions of what the nodes and relationships, what the represent and their meanings etc.
        schema_docs = """
        Link nodes represent segments of the road 
        The Link node have the following properties
        from_junction, string, the junction where the link starts from
        link_id, string, the unique id for the link
        meters, string, the length of the link in meters
        to_junction, string, the junction where the link ends at

        VMS nodes represent electronic warning signs on the road
        The VMS nodes have the following properties
        DIST_TO_UPNODE, integer, distance in meters to the up node
        EQT_EXT_ID, string, the equipment unique id number
        EQT_NO, string, the equipment unique id number
        EQT_TYPE, string, the type of equipment
        ID, integer, 3-digit id of equipment
        LATITUDE, float, latitude position of equipment
        LINK_ID, integer, unique id of link where equipment is positioned
        LONGITUDE, float, longitude position of equipment
        ROAD_CAT, string, category of road equipment is placed at
        ROAD_CODE, string, code of road equipment is placed at
        ROAD_NAME, string, name of road equipment is placed at

        RELATIONSHIPS:
        The Link node and VMS node are connected by relationship LOCATED_AT which indicates where the VMS is located at it is always VMS to Link.
        They are connected by matching identical VMS (LINK_ID) to Link (link_id),
        when matching ensure for VMS it is an integer and for Link it is a string

        The Link nodes are connected by relationship CONNECTED_TO which represents the physical road network topology.
        CONNECTED_TO relationships form bidirectional connections between adjacent road segments.
        Links are connected by matching Link (to_junction) with Link (from_junction) of adjacent segments.

        """
        # System prompt for chatbot
        system_prompt = f"""
        You are a traffic incident management expert.

        CRITICAL REQUIREMENTS - DO NOT DEVIATE:
        1. ALWAYS start by performing GetSchema tool INPUT MUST BE "" WHEN DOING SO
        2. MANDATORY SECOND STEP: Check the incident link ITSELF for VMS signs first
        3. MANDATORY THIRD STEP: Find VMS signs UPSTREAM of the incident
        4. Search exactly 50 road links upstream using CONNECTED_TO*1..50
        5. FORBIDDEN: Omnidirectional search, downstream search, or limited hop search
        6. CONTINUE: If no VMS found within 50 keep going another 20 additional hops

        STEP-BY-STEP PROCESS:
        STEP 1: Get schema with GetSchema tool
        STEP 2: Check incident link for VMS using:
        MATCH (incident:Link) WHERE incident.link_id = 'INCIDENT_LINK_ID'
        MATCH (incident)<-[:LOCATED_AT]-(vms:VMS)
        WHERE toInteger(incident.link_id) = vms.LINK_ID
        RETURN vms, 0 as distance_meters, 0 as hops_from_incident

        STEP 3: Find upstream VMS using:
        MATCH (incident:Link) WHERE incident.link_id = 'INCIDENT_LINK_ID'
        MATCH path = (incident)<-[:CONNECTED_TO*1..50]-(upstream:Link)<-[:LOCATED_AT]-(vms:VMS)
        WHERE toInteger(upstream.link_id) = vms.LINK_ID
        WITH vms, upstream, path, length(path) as hops_from_incident,
            reduce(total = 0, link IN nodes(path) | total + toInteger(coalesce(link.meters, '0'))) as distance_meters
        RETURN vms, distance_meters, hops_from_incident, upstream.link_id as vms_link_id
        ORDER BY distance_meters ASC

        CORRECT CYPHER PATTERN FOR VMS SEARCH WITH DISTANCE EXAMPLES:
        ✅ INCIDENT LINK CHECK: 
        MATCH (incident:Link) WHERE incident.link_id = '17840002118812'
        MATCH (incident)<-[:LOCATED_AT]-(vms:VMS)
        WHERE toInteger(incident.link_id) = vms.LINK_ID
        RETURN vms.EQT_NO, vms.ROAD_NAME, vms.EQT_EXT_ID, 
            0 as distance_meters, 0 as hops_from_incident

        ✅ UPSTREAM SEARCH WITH DISTANCE:
        MATCH (incident:Link) WHERE incident.link_id = '17840002118812'
        MATCH path = (incident)<-[:CONNECTED_TO*1..50]-(upstream:Link)<-[:LOCATED_AT]-(vms:VMS)
        WHERE toInteger(upstream.link_id) = vms.LINK_ID
        WITH vms, upstream, path, length(path) as hops_from_incident,
            reduce(total = 0, link IN nodes(path) | total + toInteger(coalesce(link.meters, '0'))) as distance_meters
        RETURN vms.EQT_NO, vms.ROAD_NAME, vms.EQT_EXT_ID, 
            upstream.link_id as vms_link_id, 
            distance_meters, 
            hops_from_incident
        ORDER BY distance_meters ASC

        DISTANCE CALCULATION RULES:
        - IMPORTANT: link.meters is a STRING, must convert with toInteger()
        - Use: toInteger(coalesce(link.meters, '0')) to handle null values
        - For incident link itself: distance = 0 meters, hops = 0
        - For upstream VMS: sum all converted link.meters values in the path
        - Always include distance_meters and hops_from_incident in your results
        - Order results by distance_meters ASC to show closest VMS first

        ❌ WRONG: MATCH (upstream:Link)<-[:CONNECTED_TO*1..50]-(incident:Link)
        ❌ WRONG: MATCH (incident:Link)-[:CONNECTED_TO*1..50]->(upstream:Link)
        ❌ WRONG: total + link.meters (this won't work - meters is a string!)

        DIRECTION RULE: Arrow MUST point toward the incident: upstream -> incident
        This means: (incident)<-[:CONNECTED_TO*1..50]-(upstream)

        {schema_docs}
        Use the documentation above to understand what each node/edge means

        TRAFFIC FLOW RULES:
        - Upstream = where traffic comes FROM (toward incident) 
        - Use arrow syntax: incident<-[:CONNECTED_TO*1..50]-upstream
        - This finds links where traffic flows toward the incident

        Your mission: Find upstream VMS to warn approaching drivers and prevent secondary accidents.
        Always report the distance in meters and number of hops for each VMS found.
        Use SelfReflect tool whenever you encounter challenges or need to evaluate your approach.
        Remember: link.meters is a STRING - always convert with toInteger()!
        """

        self.agent = initialize_agent(
            tools=self.road_tools,
            llm=llm,
            agent="chat-conversational-react-description",
            verbose=True,
            max_iterations=8,
            return_intermediate_steps=True,
            handle_parsing_errors=custom_parsing_error_handler,
            agent_kwargs={
                "system_message": system_prompt
            }
        )
    
    def query(self, question: str):
        try:
            response = self.agent.invoke({
                "input": question,
                "chat_history": self.chat_history
            })
            
            self.chat_history.extend([
                ("human", question),
                ("ai", response.get("output", ""))
            ])

            return response.get("output", str(response))
        except Exception as e:
            return f"Error: {e}"
    
    def clear_history(self):
        self.chat_history = []
    
    def get_history(self):
        return self.chat_history
    

chatbot = RoadNetworkChatBot()

Test run

In [10]:
chatbot.clear_history()
print("History cleared",chatbot.get_history())

response1 = chatbot.query(
    "A car accident has occurred at link_id 17840006094278 , the time is 0800, the event severity is medium, the event type is accident"
)
print("Context gathered:", response1 )



History cleared []


> Entering new AgentExecutor chain...
```json
{
    "action": "GetSchema",
    "action_input": ""
}
```
Observation: Node properties:
- **VMS**
  - `ID`: INTEGER Min: 116, Max: 227
  - `EQT_NO`: STRING Example: "E11DMSP01N"
  - `LONGITUDE`: FLOAT Min: 55.039553, Max: 55.461815
  - `DIST_TO_UPNODE`: INTEGER Min: 2, Max: 972
  - `ROAD_NAME`: STRING Example: "Sheikh Zayed Road"
  - `DIR`: FLOAT Min: 1.0, Max: 2.0
  - `ROAD_CODE`: STRING Example: "E11"
  - `EQT_TYPE`: STRING Available options: ['DMS']
  - `EQT_EXT_ID`: STRING Example: "E11DMSP01N"
  - `ROAD_CAT`: STRING Available options: ['E', 'N', 'D']
  - `LINK_ID`: INTEGER Min: 17840001512587, Max: 17840006398719
  - `LATITUDE`: FLOAT Min: 24.9017645, Max: 25.296028
- **Link**
  - `from_junction`: STRING Example: "17840007445310"
  - `link_id`: STRING Example: "17840005685950"
  - `to_junction`: STRING Example: "17840007445311"
  - `meters`: STRING Example: "43.83"
Relationship properties:
- **CONNECTED_TO**
  - `j

1. Conditional logic
2. Heuristics
3. ReAct
4. ReWOO
5. Self-reflection
6. Multiagent reasoning

## TO DO NEXT WEEK

Fix memory issue

fix agent template to get thought output
1. Prompt builder
2. Response parser

add human in loop

research about local llm for rag

LangGraph implementation ✅

